# Judge teacher labels locally (Kaggle / Colab, GPU)

Local-inference equivalent of `src/teacher_judging/judge_aux_labels.py`, for
scoring the teacher labels without an OpenRouter/OpenAI key.

It reads the JSONL produced by the labeling notebook
(`raw_primary_runs_medgemma_repaired.jsonl`), adds a `judge` block plus
`accepted_for_training` / `requires_human_audit` to each record, and writes
`judged.jsonl` in the **same shape** `apply_quality_gates.py` expects:

```
labeling notebook -> raw_primary_runs_*.jsonl
        |
        v
THIS NOTEBOOK -> judged.jsonl
        |
        v
src/teacher_judging/apply_quality_gates.py
   -> accepted_auto / human_audit_queue / rejected
        |
        v
src/teacher_labeling/build_student_sft_jsonl.py
```

### Two design choices that make a small local judge reliable
- `schema_valid` and `evidence_exact` are **computed deterministically** here,
  not asked of the model — they are mechanical checks, so this is more reliable
  than the API judge.
- The model only rates the subjective fields (`label_supported`,
  `rationale_supported`, `hallucination_risk`, `score`). Anything it omits is
  backfilled **conservatively** — missing fields push a record toward human
  review, never toward auto-accept.

### Inputs
- The labeling JSONL (attach as a Kaggle dataset or upload), e.g.
  `raw_primary_runs_medgemma_repaired.jsonl`.

### Judge model
- Default `Qwen/Qwen2.5-7B-Instruct` — ungated (no HF token needed), strong
  JSON follower, fits a T4 in 4-bit. Using a *different* model than the
  labeler also avoids a model grading its own work.
- To reuse MedGemma instead, set `JUDGE_ID = "google/medgemma-4b-it"` in
  cell 2 and provide an `HF_TOKEN` (MedGemma is gated).

In [ ]:
# === CELL 1 — install ===
import sys, subprocess, torch
cap = torch.cuda.get_device_capability(0)
print("GPU capability:", cap)
assert cap >= (7, 0), "Use a GPU with bf16 support (e.g. T4)."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "accelerate", "bitsandbytes", "huggingface_hub"], check=False)
print("install done")

In [ ]:
# === CELL 2 — config + locate input JSONL ===
import os, glob

# Default judge: ungated, strong JSON follower. Switch to MedGemma if you prefer.
JUDGE_ID = "Qwen/Qwen2.5-7B-Instruct"

# MedGemma is gated; Qwen is not. Only logs in if a token is available.
if "medgemma" in JUDGE_ID.lower() or "gemma" in JUDGE_ID.lower():
    from huggingface_hub import login
    try:
        from kaggle_secrets import UserSecretsClient
        login(UserSecretsClient().get_secret("HF_TOKEN")); print("HF login OK (secret)")
    except Exception:
        import getpass
        login(getpass.getpass("HF token (gated model): "))

# --- EDIT if your filename/location differs ---
INPUT_JSONL = sorted(glob.glob("/kaggle/input/**/*repaired*.jsonl", recursive=True)) \
    or sorted(glob.glob("/kaggle/input/**/*raw_primary_runs*.jsonl", recursive=True)) \
    or sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True)) \
    or sorted(glob.glob("./*repaired*.jsonl")) \
    or sorted(glob.glob("./raw_primary_runs_medgemma_repaired.jsonl"))
print("input jsonl candidates:", INPUT_JSONL)
assert INPUT_JSONL, "Attach/upload the labeling JSONL (output of the labeling notebook)."
INPUT_JSONL = INPUT_JSONL[0]

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_JSONL = f"{OUT_DIR}/judged.jsonl"

TEMPERATURE = 0.0
MAX_NEW_TOKENS = 512
LIMIT = 0  # 0 = judge everything; set small for a quick check
print("judge:", JUDGE_ID, "| input:", INPUT_JSONL, "| output:", OUTPUT_JSONL)

## 3. Load the judge model

In [ ]:
# === CELL 3 — load judge model ===
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

tok = AutoTokenizer.from_pretrained(JUDGE_ID)
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_ID, quantization_config=bnb, device_map="auto",
    torch_dtype=torch.bfloat16, attn_implementation="eager")
model.eval()
print("judge model loaded:", JUDGE_ID)

## 4. Judge logic

Ported from `src/teacher_judging/judge_aux_labels.py` + `prompts/judge_system.md`,
with the deterministic-vs-model split described at the top.

In [ ]:
# === CELL 4 — judge logic ===
import json

JUDGE_SYSTEM = """You are a strict rubric judge for research-only crisis triage labels.

Evaluate whether the candidate JSON is supported by the input text. Do not add new clinical advice.

Score guidance:
- 9-10: fully grounded, safe, consistent
- 7-8: usable but minor uncertainty
- 5-6: questionable, needs human review
- 0-4: reject or regenerate"""

# The model is asked ONLY for the subjective fields. schema_valid and
# evidence_exact are computed deterministically below and merged in.
JUDGE_SKELETON = """Output a single JSON object with EXACTLY these keys, nothing else:
{
  "label_supported": <true or false>,
  "rationale_supported": <true or false>,
  "hallucination_risk": <one of: low, medium, high>,
  "disagreement_risk_tier": <integer 0-3: the tier YOU would assign from the text>,
  "score": <integer 0-10 per the score guidance>,
  "reasons": [<short strings>]
}"""

LABEL_REQUIRED_KEYS = [
    "risk_tier", "confidence", "evidence_spans", "risk_factors", "protective_factors",
    "cssrs_axes", "clinical_rationale", "plain_language_summary",
    "recommended_next_step", "escalation_required", "uncertainty_flags",
]
_NEXT_STEPS = ["Supportive", "Professional support", "Urgent support",
               "Emergency escalation", "Human review"]

def compute_schema_valid(label):
    """Deterministic: does the candidate label satisfy the 11-key schema?"""
    if not isinstance(label, dict):
        return False
    if any(k not in label for k in LABEL_REQUIRED_KEYS):
        return False
    if label["risk_tier"] not in [0, 1, 2, 3]:
        return False
    if not isinstance(label["confidence"], (int, float)) or not 0 <= label["confidence"] <= 1:
        return False
    if not isinstance(label["evidence_spans"], list) or not label["evidence_spans"]:
        return False
    if not isinstance(label["escalation_required"], bool):
        return False
    if label["recommended_next_step"] not in _NEXT_STEPS:
        return False
    return True

def compute_evidence_exact(label, text):
    """Deterministic: are all evidence_spans exact substrings of the text?"""
    spans = label.get("evidence_spans") if isinstance(label, dict) else None
    if not isinstance(spans, list) or not spans:
        return False
    return all(isinstance(s, str) and s and s in text for s in spans)

def build_judge_prompt(record):
    return (
        "Evaluate this candidate auxiliary label.\n\n"
        "Input text:\n<TEXT>\n"
        f"{record['text']}\n"
        "</TEXT>\n\n"
        "Candidate JSON:\n<JSON>\n"
        f"{json.dumps(record.get('majority_label'), ensure_ascii=False, indent=2)}\n"
        "</JSON>\n\n"
        f"{JUDGE_SKELETON}"
    )

def extract_json(content):
    content = content.strip()
    if content.startswith("```"):
        content = content.strip("`")
        if content.lower().startswith("json"):
            content = content[4:].strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        start = content.find("{"); end = content.rfind("}")
        if start >= 0 and end > start:
            return json.loads(content[start:end + 1])
        raise

def repair_judge(j, label, text):
    """Merge deterministic checks + conservatively backfill missing subjective
    fields. Missing -> the cautious side (no auto-accept)."""
    if not isinstance(j, dict):
        j = {}
    j["schema_valid"] = compute_schema_valid(label)        # deterministic
    j["evidence_exact"] = compute_evidence_exact(label, text)  # deterministic
    if not isinstance(j.get("label_supported"), bool):
        j["label_supported"] = False
    if not isinstance(j.get("rationale_supported"), bool):
        j["rationale_supported"] = False
    if j.get("hallucination_risk") not in ["low", "medium", "high"]:
        j["hallucination_risk"] = "high"
    if j.get("disagreement_risk_tier") not in [0, 1, 2, 3]:
        j["disagreement_risk_tier"] = label.get("risk_tier", 0) if isinstance(label, dict) else 0
    if not isinstance(j.get("score"), int) or not 0 <= j.get("score", -1) <= 10:
        j["score"] = 5  # unknown -> 'needs human review' band
    if not isinstance(j.get("reasons"), list):
        j["reasons"] = []
    j["requires_human_audit"] = bool(j.get("requires_human_audit")) or j["score"] < 8
    return j

def already_done_ids(path):
    done = set()
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    done.add(json.loads(line)["id"])
                except (json.JSONDecodeError, KeyError):
                    pass
    return done

print("judge logic loaded")

## 5. Judging loop

Resumable: re-running skips ids already in `judged.jsonl`. Records with no
`majority_label` (hard-failed during labeling) are scored 0 and sent to human
review without calling the model.

In [ ]:
# === CELL 5 — judging loop ===
@torch.inference_mode()
def judge_one(record):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": build_judge_prompt(record)},
    ]
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
        do_sample=TEMPERATURE > 0, temperature=max(TEMPERATURE, 1e-5),
        pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

records = [json.loads(l) for l in open(INPUT_JSONL, encoding="utf-8") if l.strip()]
if LIMIT:
    records = records[:LIMIT]
print(f"records to judge: {len(records)}")

done = already_done_ids(OUTPUT_JSONL)
with open(OUTPUT_JSONL, "a", encoding="utf-8") as out:
    for index, record in enumerate(records, start=1):
        if record.get("id") in done:
            continue
        label = record.get("majority_label")
        if not label:
            record["judge"] = {"schema_valid": False, "evidence_exact": False,
                "label_supported": False, "rationale_supported": False,
                "hallucination_risk": "high", "disagreement_risk_tier": 0,
                "score": 0, "reasons": ["no majority label"],
                "requires_human_audit": True}
        else:
            try:
                content = judge_one(record)
                j = extract_json(content)
            except Exception as exc:
                j = {"reasons": [f"judge_exception:{exc}"], "requires_human_audit": True}
            record["judge"] = repair_judge(j, label, record["text"])

        judge = record.get("judge") or {}
        record["accepted_for_training"] = bool(
            label
            and judge.get("score", 0) >= 7
            and judge.get("evidence_exact", False)
            and judge.get("label_supported", False)
            and judge.get("hallucination_risk") in ["low", "medium"]
        )
        record["requires_human_audit"] = bool(
            record.get("requires_human_audit")
            or judge.get("requires_human_audit")
            or judge.get("score", 0) < 8
        )
        out.write(json.dumps(record, ensure_ascii=False) + "\n")
        out.flush()
        print(f"[{index}/{len(records)}] {record.get('id')} "
              f"score={judge.get('score')} accept={record['accepted_for_training']}")

print("done ->", OUTPUT_JSONL)

In [ ]:
# === DEBUG — inspect raw judge output for one record ===
# Run this BEFORE the full loop above if scores keep coming back as the
# fallback default (5). It prints exactly what the model said, unparsed,
# so we can see whether it's dropping the "score" key, wrapping in prose,
# or something else.
_dbg_record = records[0]
_dbg_label = _dbg_record.get("majority_label")
if not _dbg_label:
    print("record 0 has no majority_label, picking the next one with a label")
    _dbg_record = next(r for r in records if r.get("majority_label"))

_dbg_content = judge_one(_dbg_record)
print("=== RAW MODEL OUTPUT ===")
print(_dbg_content)
print()
print("=== extract_json() result ===")
try:
    print(extract_json(_dbg_content))
except Exception as exc:
    print("extract_json failed:", exc)

## 6. Quick check

In [ ]:
# === CELL 6 — sanity check ===
judged = [json.loads(l) for l in open(OUTPUT_JSONL, encoding="utf-8") if l.strip()]
accepted = sum(1 for r in judged if r.get("accepted_for_training"))
audit = sum(1 for r in judged if r.get("requires_human_audit"))
scores = [(r.get("judge") or {}).get("score") for r in judged]
dist = {s: scores.count(s) for s in sorted(set(scores), key=lambda x: (x is None, x))}
print(f"judged={len(judged)} accepted_for_training={accepted} requires_human_audit={audit}")
print("score distribution:", dist)

## 7. Hand off

Download `judged.jsonl` from `/kaggle/working`, then continue on your machine
(these stages are pure-Python, no GPU/API needed):

```bash
python src/teacher_judging/apply_quality_gates.py \
  --input  judged.jsonl \
  --output-dir data/synthetic_aux/gated_medgemma

# Gate output: accepted_auto.jsonl / human_audit_queue.jsonl / rejected.jsonl
python src/teacher_judging/export_human_audit_sheet.py \
  --input  data/synthetic_aux/gated_medgemma/human_audit_queue.jsonl \
  --output data/synthetic_aux/gated_medgemma/audit_sheet.csv

python src/teacher_labeling/build_student_sft_jsonl.py \
  --input  data/synthetic_aux/gated_medgemma/accepted_auto.jsonl \
  --output data/synthetic_aux/student_sft.jsonl
```

Note: `apply_quality_gates.py` recomputes its own accept/audit/reject decision
from the `judge` block — it does not read the `accepted_for_training` /
`requires_human_audit` fields this notebook writes (those are just convenience
flags). To reach `accepted_auto`, a record needs `score >= audit-score` (8),
deterministic `evidence_exact`, `label_supported`, non-high hallucination risk,
and no tier-3 / escalation / teacher-disagreement reasons; everything else goes
to `human_audit_queue.jsonl`.